# CrewAI — Simple Agent

**Framework:** CrewAI  
**Level:** 01 — Simple  
**Model:** `gemini-2.0-flash` via CrewAI's LLM abstraction

### What You Will Learn
- CrewAI's **role-playing model**: agents have a role, goal, and backstory (not just a system prompt)
- How to define tools with the `@tool` decorator from `crewai.tools`
- How `Task` separates the *what* (description) from the *who* (agent) from the *output spec*
- How `Crew` orchestrates agents and tasks into a runnable workflow

## Concept: The Role-Playing Model

```
┌─────────────────────────────────────────┐
│                  CREW                   │
│                                         │
│  ┌───────────────────────────────────┐  │
│  │  Agent: Travel Weather Assistant  │  │
│  │  role     = "Weather Expert"      │  │  ← who the agent IS
│  │  goal     = "Give accurate info"  │  │  ← what it's optimizing for
│  │  backstory= "Experienced concierge│  │  ← shapes its reasoning style
│  │  tools    = [get_weather,get_time]│  │
│  └───────────────────────────────────┘  │
│                    │                    │
│  ┌─────────────────▼─────────────────┐  │
│  │  Task: Answer weather query       │  │
│  │  description     = "Find weather  │  │  ← WHAT to do
│  │                     for {city}"   │  │
│  │  expected_output = "2–3 sentence  │  │  ← defines DONE
│  │                     summary"      │  │
│  └───────────────────────────────────┘  │
│                                         │
│  Process: Sequential                    │
└─────────────────────────────────────────┘
               │
               ▼
         crew.kickoff() → result
```

**Key insight:** CrewAI is designed around **human team metaphors**. You describe agents the way you'd describe a team member in a job posting — their role, what they're trying to achieve, and their background. This makes multi-agent setups very readable (see 02-Architectures/).

**Compare to ADK/LangChain:** Those use a `system prompt` / `instruction`. CrewAI splits that into `role + goal + backstory`, which provides richer context to the LLM — the agent genuinely 'plays' the role.

## Setup

In [ ]:
import os
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY not set — check your .env file"
print("✓ GOOGLE_API_KEY loaded")

## Tool Definitions

CrewAI's `@tool` decorator takes a **tool name string** as an argument (unlike LangChain which infers it from the function name).  
The docstring still serves as the tool description the LLM reads.  
**Important:** CrewAI tool return values should be strings (or easily serializable) — the LLM reads them as text.

In [ ]:
@tool("Weather Tool")
def get_weather(city: str) -> str:
    """Get the current weather conditions for a given city.
    Returns temperature, weather condition, and humidity.
    Input: city name (string).
    """
    mock_data = {
        "london":    ("Cloudy", 12, 78),
        "new york":  ("Sunny",  22, 45),
        "bangalore": ("Rainy",  25, 85),
        "tokyo":     ("Clear",  18, 60),
    }
    key = city.lower()
    if key in mock_data:
        condition, temp, humidity = mock_data[key]
        return (f"{city}: {condition}, {temp}°C, humidity {humidity}%. "
                f"{'Bring an umbrella!' if condition == 'Rainy' else 'Enjoy the weather!'}")
    return f"No weather data for '{city}'. Available: London, New York, Bangalore, Tokyo."


@tool("Time Tool")
def get_time(city: str) -> str:
    """Get the current local time for a given city.
    Input: city name (string).
    """
    mock_times = {
        "london":    "14:30 GMT",
        "new york":  "09:30 EST",
        "bangalore": "20:00 IST",
        "tokyo":     "22:30 JST",
    }
    key = city.lower()
    if key in mock_times:
        return f"Current time in {city}: {mock_times[key]}"
    return f"No time data for '{city}'. Available: London, New York, Bangalore, Tokyo."


# Quick test
print(get_weather.run("Bangalore"))
print(get_time.run("Tokyo"))

## Agent Definition

The three-part persona system:
- **`role`** — the agent's job title; used in logs and multi-agent delegation
- **`goal`** — what the agent is optimizing for (shapes its decisions)
- **`backstory`** — narrative context that influences tone and depth of reasoning

In [ ]:
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)

weather_agent = Agent(
    role="Travel Weather Assistant",
    goal="Provide accurate, friendly weather and time information for any city the traveler asks about.",
    backstory=(
        "You are an experienced travel concierge who helps travelers plan their trips "
        "by providing up-to-date local weather conditions and time zone information. "
        "You always use your tools to get the latest data before answering."
    ),
    tools=[get_weather, get_time],
    llm=gemini,
    verbose=True,
)

print(f"Agent created: '{weather_agent.role}'")

## Task Definition

`Task` separates concerns:
- **`description`** — what to do (the instruction, accepts format variables)
- **`expected_output`** — what 'done' looks like (the LLM uses this to self-assess quality)
- **`agent`** — which agent does it

This separation makes it easy to reuse the same agent with different tasks, or swap agents on the same task.

In [ ]:
def create_task(query: str) -> Task:
    return Task(
        description=f"Answer the following user question about weather or time: {query}",
        expected_output=(
            "A concise, friendly response that directly answers the user's question. "
            "Include specific numbers (temperature, humidity) where available. "
            "Keep it to 2–3 sentences."
        ),
        agent=weather_agent,
    )

print("Task factory ready")

## Run the Crew

In [ ]:
def run(query: str) -> str:
    task = create_task(query)
    crew = Crew(
        agents=[weather_agent],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
    )
    return str(crew.kickoff())


# Query 1
query1 = "What's the weather like in Bangalore right now?"
print(f"User: {query1}")
print(f"Agent: {run(query1)}")

In [ ]:
# Query 2
query2 = "What time is it in Tokyo, and is it a good day to go outside?"
print(f"User: {query2}")
print(f"Agent: {run(query2)}")

In [ ]:
# Run with verbose=True to see CrewAI's detailed thinking trace
task = create_task("What's the weather in London and what time is it there?")
crew_verbose = Crew(
    agents=[weather_agent],
    tasks=[task],
    process=Process.sequential,
    verbose=True,  # shows agent's chain-of-thought and tool calls
)
result = crew_verbose.kickoff()
print(f"\nFinal answer: {result}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `role + goal + backstory` | Richer persona than a single system prompt; shapes reasoning quality |
| `@tool("Tool Name")` | Named tools; return strings for clean LLM consumption |
| `Task.expected_output` | Tells the agent what 'done' looks like; acts as an implicit quality check |
| `Process.sequential` | One task at a time; `Process.hierarchical` for manager-worker patterns |
| `crew.kickoff()` | Triggers the full workflow; returns a `CrewOutput` object |

### All Four Frameworks — Simple Level Comparison

| Dimension | ADK | LangChain | LangGraph | CrewAI |
|---|---|---|---|---|
| Agent persona | `instruction=` | `system` prompt | `SystemMessage` | `role + goal + backstory` |
| Tool decorator | Plain function | `@tool` | `@tool` | `@tool("Name")` |
| Run method | `runner.run_async()` | `executor.invoke()` | `graph.invoke()` | `crew.kickoff()` |
| Output | Event stream | Dict `{output}` | State dict | `CrewOutput` |
| Loop visibility | Hidden | Hidden | **Explicit graph** | Hidden |

### What Changes at 02-Intermediate
- Multiple agents working together (e.g., Researcher + Writer)
- Shared crew context and task delegation
- Structured output with Pydantic `output_pydantic`
- Task dependencies (`context=[previous_task]`)

### Next Notebook
[02-Intermediate →](../02-intermediate/agent.ipynb)